# General

For more informations, si the documentation *LLM and GenAI*.

# Import & Configs

In [24]:
import json

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [23]:
%load_ext autoreload
%autoreload 2

from src.retrieval.retriever import MedicalRetriever
from src.pipeline.medical_assistant import ask_medical_assistant

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Questions Set

In [28]:
with open(
    "../data/evaluations/gold_questions.json"
) as f:

    questions = json.load(f)

questions

[{'question': 'What is glioblastoma?'},
 {'question': 'How is glioblastoma prognosis evaluated?'},
 {'question': 'What MRI techniques are used for glioblastoma?'},
 {'question': 'What is peritumoral edema?'},
 {'question': 'What is pseudoprogression?'},
 {'question': 'How is tumor progression detected?'},
 {'question': 'What biomarkers are associated with glioblastoma?'},
 {'question': 'What are the limitations of MRI in glioma diagnosis?'}]

# Retrieval Evaluation

In [6]:
retriever = MedicalRetriever()

/home/jeremy/Documents/dev/LLM_RAG/Medical_assistant/.ma_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
for question in questions:
    print(f'{question:-^80}')
    results = retriever.retrieve(question)
    print('\n')
    for key, val in results.items():
        #print(f'--- {key}:')
        if key == 'documents':
            for i, doc in enumerate(val[0]):
                print(f'[{results['ids'][0][i]}]\n{doc}\n')
        else:
            #print(f'{val}')
            pass
        #print('\n')
    print("\n\n")

-----------------------------What is glioblastoma?------------------------------
Original query: What is glioblastoma?
Processed query: glioblastoma


[42135047_3]
Brain gliomas are among the most common primary brain tumors of the central nervous system, encompassing a wide spectrum of tumor grades and diverse biological behaviors. Key aspects of their clinical management include accurate diagnosis and grading, tumor extent delineation, preoperative evaluation and radiation therapy target planning, as well as differentiation between post-treatment recurrence and treatment-related changes.

[42000417_0]
Glioblastoma IDH-wildtype: An integrative review of pathophysiological mechanisms, diagnostic innovations, and emerging therapeutic modalities. The most dangerous and fatal primary brain tumor in adults is glioblastoma multiforme/IDH wild-type glioblastoma (GBM), which is progressive, diffuse, and unresponsive to conventional treatment.

[42000417_1]
The most dangerous and fatal primary

| Question  | Retrieval OK ? | Comments |
|-----|:-----:|-----|
| Q1 | 4 |  |
| Q2 | 3 |  |
| Q3 | 2 |  |
| Q4 | 3 | Confusion btw glioma & glioblastoma |
| Q5 | 2 | Relevance ? | 
| Q6 | 3 |  |
| Q7 | 2 | Only doc 1 is relevant |
| Q8 | 2 | Doc 1 : not relevant ||
| Q9 | 1 | Not Relevant | 
| Q10 | 1 | Confusion btw glioma & glioblastoma |

With:
- 1 = bad
- 5 = excellent

Identified Limitations:
- Chunks that are too specialized.
- Chunks at the wrong level
- Diversification issues

# Answer Evaluation

In [31]:
results = []

for item in questions:
    
    question = item["question"]
    print(f'{question:-^80}')

    output = ask_medical_assistant(
        question,
        retriever
    )

    results.append(output)
    print("\n\n")

-----------------------------What is glioblastoma?------------------------------
Original query: What is glioblastoma?
Processed query: glioblastoma
Glioblastoma is a type of primary brain tumor that is highly malignant and aggressive. It is characterized by its rapid progression, diffuse nature, and resistance to conventional treatments such as surgery, radiation therapy, and chemotherapy. The most dangerous and fatal primary brain tumor in adults, glioblastoma multiforme/IDH wild-type glioblastoma (GBM), is progressive, diffuse, and unresponsive to conventional treatment.Sources used:
[1] [Clinical practice guideline for integrated PET/MRI in brain gliomas(2026 edition)] (2026) PMID:42135047
[2] Glioblastoma IDH-wildtype: An integrative review of pathophysiological mechanisms, diagnostic innovations, and emerging therapeutic modalities (2026) PMID:42000417
[3] Glioblastoma IDH-wildtype: An integrative review of pathophysiological mechanisms, diagnostic innovations, and emerging thera

| Question  | Relevance | Faithfulness | Clarity |
|:-----|:-----:|:-----:|:-----:|
| Q1 | ? | ? | ? |
| Q2 | ? | ? | ? |
| Q3 | ? | ? | ? |
| Q4 | ? | ? | ? |
| Q5 |  ? | ? | ? |
| Q6 |  ? | ? | ? |
| Q7 |  ? | ? | ? |
| Q8 |  ? | ? | ? |
| Q9 |  ? | ? | ? |
| Q10 |  ? | ? | ? |

With:
- 1 = bad
- 5 = excellent